# GymRAVANA model training and evaluation

This notebook compares transparent tabular classifiers for the **trainer-recorded progression-readiness** target. It does not fabricate labels, silently replace the target with a public-dataset outcome, or export a model. With insufficient genuine data, every training section is skipped.

## Preconditions

Run `php artisan gymravana:export-readiness-data` from the Laravel project root. Complete Notebook 01 first and investigate every schema, privacy, duplicate, missing-value and class-balance warning. The thresholds below are minimum engineering gates for attempting an academic prototype; passing them does not prove that the sample is representative or that a model is safe for real decisions.

In [ ]:
from pathlib import Path
from hashlib import sha256
import json

import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, make_scorer,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
working_directory = Path.cwd().resolve()
if working_directory.name == 'notebooks':
    PROJECT_ROOT = working_directory.parent.parent
elif (working_directory / 'ai').is_dir():
    PROJECT_ROOT = working_directory
else:
    raise RuntimeError('Open this notebook from the GymRAVANA project or ai/notebooks directory.')

DATASET_PATH = PROJECT_ROOT / 'ai' / 'data' / 'readiness_dataset.csv'
METADATA_PATH = PROJECT_ROOT / 'ai' / 'data' / 'readiness_dataset.metadata.json'
SELECTION_REPORT_PATH = PROJECT_ROOT / 'ai' / 'artifacts' / 'model_selection.json'
print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset: {DATASET_PATH}')

In [ ]:
if not DATASET_PATH.exists():
    raise FileNotFoundError('Dataset not found. Run: php artisan gymravana:export-readiness-data')
if not METADATA_PATH.exists():
    raise FileNotFoundError('Dataset metadata not found. Export the dataset again before training.')

df = pd.read_csv(DATASET_PATH)
metadata = json.loads(METADATA_PATH.read_text(encoding='utf-8'))
actual_hash = sha256(DATASET_PATH.read_bytes()).hexdigest()
assert metadata.get('schema_version') == 1, 'Unsupported readiness dataset schema. Export it again with the current Laravel application.'
assert metadata.get('dataset_sha256') == actual_hash, 'CSV fingerprint does not match its metadata. Do not train on an altered or mismatched export.'
assert metadata.get('row_count') == len(df), 'Metadata row count does not match the CSV.'
assert metadata.get('columns') == list(df.columns), 'Metadata columns do not match the CSV header.'
assert metadata.get('target') == 'ready_for_progression', 'Unexpected target in dataset metadata.'
print(f'Rows: {len(df)} | Columns: {len(df.columns)}')
print(json.dumps(metadata, indent=2))

In [ ]:
NUMERIC_FEATURES = [
    'workout_completions', 'wellness_completions',
    'trainer_sessions_scheduled', 'trainer_sessions_completed',
    'attendance_rate', 'cancelled_or_declined_sessions',
    'active_days', 'consistency_rate', 'activity_points',
    'previous_goal_completion', 'previous_rating',
    'workout_change', 'consistency_change',
]
CATEGORICAL_FEATURES = ['previous_assessment']
MODEL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = 'ready_for_progression'
GROUP_COLUMN = 'member_key'
AUDIT_ONLY_COLUMNS = ['observation_month', 'label_recorded_at']
REQUIRED_COLUMNS = set(MODEL_FEATURES + [TARGET, GROUP_COLUMN] + AUDIT_ONLY_COLUMNS)
FORBIDDEN_COLUMNS = {
    'user_id', 'trainer_profile_id', 'name', 'email', 'phone',
    'weight_kg', 'height_cm', 'waist_cm', 'chest_cm',
    'trainer_notes', 'readiness_rationale', 'therapy_request', 'diagnosis',
}

assert not (set(MODEL_FEATURES) & {TARGET, GROUP_COLUMN, *AUDIT_ONLY_COLUMNS})
print({'numeric_features': len(NUMERIC_FEATURES), 'categorical_features': CATEGORICAL_FEATURES})

In [ ]:
missing_columns = sorted(REQUIRED_COLUMNS - set(df.columns))
forbidden_columns = sorted(set(df.columns) & FORBIDDEN_COLUMNS)
assert not missing_columns, f'Missing required columns: {missing_columns}'
assert not forbidden_columns, f'Sensitive or leakage-prone columns found: {forbidden_columns}'

for column in NUMERIC_FEATURES + [TARGET]:
    df[column] = pd.to_numeric(df[column], errors='coerce')

observed_targets = set(df[TARGET].dropna().astype(int).unique().tolist())
assert observed_targets <= {0, 1}, f'Unexpected target values: {sorted(observed_targets)}'
if not df.empty:
    assert df[TARGET].notna().all(), 'Every exported row must have a readiness label.'
    assert df[GROUP_COLUMN].notna().all(), 'Every row needs a pseudonymous member group.'

duplicate_count = int(df.duplicated(subset=[GROUP_COLUMN, 'observation_month', 'label_recorded_at']).sum()) if not df.empty else 0
conflicting_member_months = int((df.groupby([GROUP_COLUMN, 'observation_month'])[TARGET].nunique() > 1).sum()) if not df.empty else 0
assert duplicate_count == 0, f'Duplicate observation keys found: {duplicate_count}'
assert conflicting_member_months == 0, 'Contradictory readiness labels exist for the same member and observation month. Investigate them before training.'
print({'targets': sorted(observed_targets), 'duplicates': duplicate_count, 'conflicting_member_months': conflicting_member_months, 'forbidden_columns': forbidden_columns})

## Public dataset decision

The reviewed Kaggle and Hugging Face candidates do not have an equivalent trainer-readiness target and acceptable feature contract. They are therefore not merged into this dataframe. See `docs/ai/public-dataset-suitability.md` for the provenance audit.

In [ ]:
MIN_ROWS = 30
MIN_ROWS_PER_CLASS = 10
MIN_MEMBER_GROUPS = 10
MIN_GROUPS_PER_CLASS = 5

class_counts_series = df[TARGET].value_counts() if not df.empty else pd.Series(dtype='int64')
groups_per_class_series = df.groupby(TARGET)[GROUP_COLUMN].nunique() if not df.empty else pd.Series(dtype='int64')
class_counts = {str(int(label)): int(count) for label, count in class_counts_series.items()}
groups_per_class = {str(int(label)): int(count) for label, count in groups_per_class_series.items()}
member_group_count = int(df[GROUP_COLUMN].nunique()) if not df.empty else 0

gate_reasons = []
if len(df) < MIN_ROWS:
    gate_reasons.append(f'need at least {MIN_ROWS} labeled rows; found {len(df)}')
if set(class_counts) != {'0', '1'}:
    gate_reasons.append('both ready and not-ready classes are required')
elif min(class_counts.values()) < MIN_ROWS_PER_CLASS:
    gate_reasons.append(f'each class needs at least {MIN_ROWS_PER_CLASS} rows')
if member_group_count < MIN_MEMBER_GROUPS:
    gate_reasons.append(f'need at least {MIN_MEMBER_GROUPS} distinct member groups; found {member_group_count}')
if set(groups_per_class) == {'0', '1'} and min(groups_per_class.values()) < MIN_GROUPS_PER_CLASS:
    gate_reasons.append(f'each class needs labels from at least {MIN_GROUPS_PER_CLASS} member groups')

training_allowed = not gate_reasons
gate_report = {
    'training_allowed': training_allowed,
    'rows': len(df),
    'class_counts': class_counts,
    'member_groups': member_group_count,
    'groups_per_class': groups_per_class,
    'reasons': gate_reasons,
}
print(json.dumps(gate_report, indent=2))

## Leakage-safe holdout

Rows from one member must never appear in both training and test sets. Multiple grouped candidates are inspected only to find a holdout containing both classes; the fixed random seed keeps this reproducible. The member key and timestamps remain audit/grouping fields, never model inputs.

In [ ]:
train_df = pd.DataFrame(columns=df.columns)
test_df = pd.DataFrame(columns=df.columns)

if training_allowed:
    X = df[MODEL_FEATURES]
    y = df[TARGET].astype(int)
    groups = df[GROUP_COLUMN]
    splitter = GroupShuffleSplit(n_splits=20, test_size=0.2, random_state=RANDOM_STATE)
    selected_indices = None
    for train_index, test_index in splitter.split(X, y, groups=groups):
        if set(y.iloc[train_index].unique()) == {0, 1} and set(y.iloc[test_index].unique()) == {0, 1}:
            selected_indices = (train_index, test_index)
            break

    if selected_indices is None:
        training_allowed = False
        gate_reasons.append('no grouped 80/20 holdout retained both classes')
        print('Training skipped:', gate_reasons[-1])
    else:
        train_index, test_index = selected_indices
        train_df = df.iloc[train_index].copy()
        test_df = df.iloc[test_index].copy()
        overlap = set(train_df[GROUP_COLUMN]) & set(test_df[GROUP_COLUMN])
        assert not overlap, 'Member leakage detected between train and test sets.'
        print({
            'train_rows': len(train_df), 'test_rows': len(test_df),
            'train_members': train_df[GROUP_COLUMN].nunique(),
            'test_members': test_df[GROUP_COLUMN].nunique(),
            'member_overlap': len(overlap),
        })
else:
    print('Grouped holdout skipped because the evidence gate did not pass.')

In [ ]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median', keep_empty_features=True)),
    ('scaler', StandardScaler()),
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('encoder', OneHotEncoder(handle_unknown='ignore')),
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, NUMERIC_FEATURES),
    ('categorical', categorical_pipeline, CATEGORICAL_FEATURES),
])

candidate_models = {
    'logistic_regression': LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE,
    ),
    'random_forest': RandomForestClassifier(
        n_estimators=200, min_samples_leaf=2, class_weight='balanced_subsample',
        random_state=RANDOM_STATE, n_jobs=-1,
    ),
}

def build_pipeline(estimator):
    return Pipeline([
        ('preprocessing', clone(preprocessor)),
        ('classifier', clone(estimator)),
    ])

print('Candidates:', list(candidate_models))
print('XGBoost is deferred: two explainable baselines are sufficient until real sample size justifies added complexity.')

In [ ]:
holdout_rows = []
trained_models = {}

if training_allowed:
    X_train = train_df[MODEL_FEATURES]
    y_train = train_df[TARGET].astype(int)
    X_test = test_df[MODEL_FEATURES]
    y_test = test_df[TARGET].astype(int)

    for model_name, estimator in candidate_models.items():
        pipeline = build_pipeline(estimator)
        pipeline.fit(X_train, y_train)
        predictions = pipeline.predict(X_test)
        probabilities = pipeline.predict_proba(X_test)[:, 1]
        matrix = confusion_matrix(y_test, predictions, labels=[0, 1])
        holdout_rows.append({
            'model': model_name,
            'accuracy': accuracy_score(y_test, predictions),
            'precision': precision_score(y_test, predictions, zero_division=0),
            'recall': recall_score(y_test, predictions, zero_division=0),
            'f1': f1_score(y_test, predictions, zero_division=0),
            'roc_auc': roc_auc_score(y_test, probabilities),
            'confusion_matrix_tn_fp_fn_tp': matrix.ravel().tolist(),
        })
        trained_models[model_name] = pipeline

    holdout_results = pd.DataFrame(holdout_rows).sort_values('f1', ascending=False)
    display(holdout_results)
else:
    holdout_results = pd.DataFrame(columns=[
        'model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc',
        'confusion_matrix_tn_fp_fn_tp',
    ])
    print('No estimator was fitted and no holdout metrics were produced.')

In [ ]:
cv_rows = []
cv_results = pd.DataFrame()

if training_allowed:
    X = df[MODEL_FEATURES]
    y = df[TARGET].astype(int)
    groups = df[GROUP_COLUMN]
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_splits = list(cv.split(X, y, groups=groups))
    valid_folds = all(
        set(y.iloc[train_index].unique()) == {0, 1}
        and set(y.iloc[test_index].unique()) == {0, 1}
        for train_index, test_index in cv_splits
    )

    if not valid_folds:
        print('Grouped cross-validation skipped because at least one fold lost a class.')
    else:
        scoring = {
            'accuracy': 'accuracy',
            'precision': make_scorer(precision_score, zero_division=0),
            'recall': make_scorer(recall_score, zero_division=0),
            'f1': make_scorer(f1_score, zero_division=0),
            'roc_auc': 'roc_auc',
        }
        for model_name, estimator in candidate_models.items():
            scores = cross_validate(
                build_pipeline(estimator), X, y, cv=cv_splits,
                scoring=scoring, error_score='raise', n_jobs=1,
            )
            cv_rows.append({
                'model': model_name,
                **{f'mean_{metric}': float(scores[f'test_{metric}'].mean()) for metric in scoring},
                **{f'std_{metric}': float(scores[f'test_{metric}'].std()) for metric in scoring},
            })
        cv_results = pd.DataFrame(cv_rows).sort_values(['mean_f1', 'mean_recall'], ascending=False)
        display(cv_results)
else:
    print('Cross-validation skipped because the evidence gate did not pass.')

In [ ]:
if not cv_results.empty:
    selected_model_name = str(cv_results.iloc[0]['model'])
    print(f'Provisional candidate: {selected_model_name}')
    print('Selection is based on grouped mean F1, then recall. Review fold variance and errors before accepting it.')

    selection_report = {
        'report_version': 1,
        'dataset_schema_version': metadata['schema_version'],
        'dataset_sha256': actual_hash,
        'dataset_rows': int(len(df)),
        'selected_model': selected_model_name,
        'selection_rule': 'highest grouped mean F1, then grouped mean recall',
        'random_state': RANDOM_STATE,
        'model_features': MODEL_FEATURES,
        'holdout_results': json.loads(holdout_results.to_json(orient='records')),
        'cross_validation_results': json.loads(cv_results.to_json(orient='records')),
        'gate_report': gate_report,
    }
    SELECTION_REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    SELECTION_REPORT_PATH.write_text(json.dumps(selection_report, indent=2), encoding='utf-8')
    print(f'Selection report written: {SELECTION_REPORT_PATH}')
else:
    selected_model_name = None
    print('No model selected. This is the correct outcome until genuine evidence supports comparison.')
    print('No model-selection report was written.')

print('No serialized model artifact is exported by Notebook 02.')

## Gate before Notebook 03

Do not create explainability claims or export a production artifact until Notebook 02 has run on genuine, reviewed labels and its grouped holdout and cross-validation results are credible. When all gates pass, this notebook writes an ignored `model_selection.json` handoff bound to the exact dataset fingerprint. Notebook 03 must verify that report, explain the chosen model, record limitations and export its feature schema and metrics together with the model.